# step05_significant_revisions — Significant Revision Events and Content Changes

Build a reproducible candidate sample and retrieve parent/revision diffs where network access permits.

Run cells from top to bottom. Shared helpers and file paths are in `analysis_common.py`. Outputs go to `analysis_results/` and `figures/`.

In [1]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis' / 'analysis_common.py').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis'))
from analysis_common import *
df=load_revisions().sort_values(['article_title','timestamp','revision_id'])
monthly=monthly_activity(df); spikes=robust_spikes(monthly); spike_rows=spikes[spikes.is_spike].nlargest(12,'revisions')
size_rows=df.dropna(subset=['size_change_bytes']).reindex(df.dropna(subset=['size_change_bytes']).size_change_bytes.abs().nlargest(12).index)
revert_mask=df.edit_comment.str.contains(r'\b(revert|reverted|undo|undid|rollback|rvv?)\b',case=False,regex=True,na=False)
comment_rows=df[revert_mask].nlargest(8,'revision_id')
# Stratified sample: large size change, high-spike months, comment/revert signals, early and recent dates.
candidates=pd.concat([size_rows,comment_rows,df[df.article_title.isin(spike_rows.article_title)].groupby('article_title',group_keys=False).tail(2),df.groupby('article_title',group_keys=False).head(1),df.groupby('article_title',group_keys=False).tail(1)]).drop_duplicates('revision_id').dropna(subset=['revision_id','parent_id'])
candidates=candidates.sort_values('size_change_bytes',key=lambda x:x.abs(),ascending=False).drop_duplicates('article_title').head(30).copy()
candidates['candidate_reason']=np.select([candidates.edit_comment.str.contains(r'\b(revert|undo|rollback|rv)\b',case=False,regex=True),candidates.size_change_bytes.abs().ge(10000)],[ 'Edit summary suggests revert; verify diff','Large observed byte change'],default='Stratified temporal/activity sample')
cols=['article_title','timestamp','revision_id','parent_id','username','edit_comment','article_size_bytes','size_change_bytes','candidate_reason']
save_table(candidates[cols],'step05_significant_revision_candidates.csv'); display(candidates[cols])


C:\Users\Jackson\AppData\Local\Temp\ipykernel_29768\591042374.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  revert_mask=df.edit_comment.str.contains(r'\b(revert|reverted|undo|undid|rollback|rvv?)\b',case=False,regex=True,na=False)
C:\Users\Jackson\AppData\Local\Temp\ipykernel_29768\591042374.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  candidates['candidate_reason']=np.select([candidates.edit_comment.str.contains(r'\b(revert|undo|rollback|rv)\b',case=False,regex=True),candidates.size_change_bytes.abs().ge(10000)],[ 'Edit summary suggests revert; verify diff','Large observed byte change'],default='Stratified temporal/activity sample')


,article_title,timestamp,revision_id,parent_id,username,edit_comment,article_size_bytes,size_change_bytes,candidate_reason
201572,Russian Revolution,2005-10-25 19:23:47+00:00,26464652,26464358,Ritchy,rv vandalism to last version by Wikibofh,22662,-3537205.0,Edit summary suggests revert; verify diff
267007,World War I,2005-12-08 13:02:16+00:00,30587516,30567121,203.218.37.105,/* Causes */,2716074,2642330.0,Large observed byte change
38231,Assassination of John F. Kennedy,2007-02-19 21:05:08+00:00,109385280,109384443,12.169.6.40,Removed glut of nonsense text,46974,-1716907.0,Large observed byte change
290446,World War II,2005-12-23 03:19:58+00:00,32441095,32440959,202.75.0.129,/* Participants */,1613424,1524224.0,Large observed byte change
229018,September 11 attacks,2026-11-08 04:05:24+00:00,1384420834,1384180652,گ-ژ-چ-پ-ڤ,/* See also */,327879,183.0,Stratified temporal/activity sample
254420,Vietnam War,2026-11-06 21:54:36+00:00,1384155769,1384088934,Emiya1980,Undid revision [[Special:Diff/1384088934|13840...,283279,174.0,Stratified temporal/activity sample
116557,Emancipation Proclamation,2026-11-10 06:27:03+00:00,1384830413,1384718356,InternetArchiveBot,Rescuing 1 sources and tagging 0 as dead. #IAB...,134638,72.0,Stratified temporal/activity sample
14929,American Civil War,2026-11-09 17:51:56+00:00,1384701279,1384162791,LightningBag,"""50,000 free civilians killed"" claim not suppo...",242186,-52.0,Stratified temporal/activity sample
64938,Battle of Gettysburg,2026-11-06 01:40:28+00:00,1383995269,1383994754,CWenger,Undid revision [[Special:Diff/1383993517|13839...,134456,-44.0,Stratified temporal/activity sample
146979,Gulf War,2026-11-09 07:38:32+00:00,1384638139,1384637879,NesserWiki,/* External links */ Added [[Category:Wars inv...,262709,36.0,Stratified temporal/activity sample


In [2]:
# Retrieval is attempted only for a small, diverse sample. API responses are stored with IDs for auditability.
import requests, time, html
API='https://en.wikipedia.org/w/api.php'
retrieved=[]; network_available=True
for row in candidates.head(8).itertuples():
    rec={'article_title':row.article_title,'timestamp':str(row.timestamp),'revision_id':int(row.revision_id),'parent_id':int(row.parent_id),'edit_comment':row.edit_comment,'size_change_bytes':row.size_change_bytes,'retrieval_status':'not attempted','diff_html':''}
    if not network_available: rec['retrieval_status']='skipped after network/API failure on earlier request'
    else:
        try:
            response=requests.get(API,params={'action':'compare','fromrev':rec['parent_id'],'torev':rec['revision_id'],'format':'json','formatversion':2},headers={'User-Agent':'WikipediaDigitalArchaeologyResearch/1.0 (educational analysis)'},timeout=5)
            response.raise_for_status(); payload=response.json(); comp=payload.get('compare',{})
            if comp.get('body') is not None: rec['diff_html']=comp['body']; rec['retrieval_status']='retrieved'
            else: rec['retrieval_status']='API returned no diff: '+str(payload.get('errors',payload))[:400]
        except Exception as exc: rec['retrieval_status']=f'retrieval failed: {type(exc).__name__}: {exc}'; network_available=False
    retrieved.append(rec); time.sleep(1.0)
diffs=pd.DataFrame(retrieved); save_table(diffs,'step05_revision_diffs_raw.csv'); display(diffs[['article_title','timestamp','revision_id','parent_id','retrieval_status']])


,article_title,timestamp,revision_id,parent_id,retrieval_status
0,Russian Revolution,2005-10-25 19:23:47+00:00,26464652,26464358,retrieved
1,World War I,2005-12-08 13:02:16+00:00,30587516,30567121,retrieved
2,Assassination of John F. Kennedy,2007-02-19 21:05:08+00:00,109385280,109384443,retrieved
3,World War II,2005-12-23 03:19:58+00:00,32441095,32440959,retrieved
4,September 11 attacks,2026-11-08 04:05:24+00:00,1384420834,1384180652,retrieved
5,Vietnam War,2026-11-06 21:54:36+00:00,1384155769,1384088934,retrieved
6,Emancipation Proclamation,2026-11-10 06:27:03+00:00,1384830413,1384718356,retrieved
7,American Civil War,2026-11-09 17:51:56+00:00,1384701279,1384162791,retrieved


In [3]:
# Strip HTML tags for a compact, reviewable text excerpt; keep raw HTML in the companion CSV.
import re
def plain_diff(s):
    s=re.sub(r'<\s*br\s*/?>','\n',s,flags=re.I); s=re.sub(r'<[^>]+>',' ',s); return html.unescape(re.sub(r'\s+',' ',s)).strip()
examples=diffs.copy(); examples['diff_excerpt']=examples.diff_html.map(plain_diff).str.slice(0,2500)
examples['change_classification']='Requires human review; diff excerpt alone does not establish historical significance'
examples['reviewer_notes']=''
save_table(examples.drop(columns='diff_html'),'step05_revision_case_studies.csv'); display(examples[['article_title','revision_id','parent_id','retrieval_status','diff_excerpt']].head(10))


,article_title,revision_id,parent_id,retrieval_status,diff_excerpt
0,Russian Revolution,26464652,26464358,retrieved,Line 1: Line 1: {{History_of_Russia}} The '''R...
1,World War I,30587516,30567121,retrieved,Line 50: Line 50: __TOC__ __TOC__ FUCK!!!FUCK!...
2,Assassination of John F. Kennedy,109385280,109384443,retrieved,Line 32: Line 32: Vice-President Johnson (who ...
3,World War II,32441095,32440959,retrieved,Line 55: Line 55: kjhjk kjhjk AAAAAAAAAAAAAAAA...
4,September 11 attacks,1384420834,1384180652,retrieved,Line 676: Line 676: ** [[Attack on Pearl Harbo...
5,Vietnam War,1384155769,1384088934,retrieved,Line 4: Line 4: {{Use American English|date=De...
6,Emancipation Proclamation,1384830413,1384718356,retrieved,"Line 322: Line 322: * Crowther, Edward R. ""Ema..."
7,American Civil War,1384701279,1384162791,retrieved,Line 75: Line 75: }} }} | casualties3 = {{plai...


## Classification and evidence

Inspect the rendered diff excerpts and raw API HTML before assigning categories such as additions, removals, framing changes, references, or reverts. The retrieval cell records failures (for example, revision visibility or network restrictions) instead of fabricating content. Edit summaries are leads, not verified descriptions. A proposed revert is verified only when the diff supports it. The case-study table has a reviewer-notes column for documented examples.

In [4]:
# Preserve added/deleted direction and direct diff links in the case-study evidence.
import re, html
from urllib.parse import quote
def diff_lines(markup, kind):
    blocks = re.findall(r'<td[^>]*diff-' + kind + r'[^>]*>(.*?)</td>', str(markup), flags=re.S)
    clean = []
    for block in blocks:
        block = re.sub(r'<br\s*/?>', ' ', block, flags=re.I)
        clean.append(html.unescape(re.sub(r'<[^>]+>', '', block)).strip())
    return ' | '.join(x for x in clean if x)
examples = diffs.copy()
def useful_excerpt(s):
    s=str(s); low=s.lower()
    markers=['you are really fat','fuck!!!','bsadf82q','aaaaaaaaaaaaaaaa','sadfsadf']
    for marker in markers:
        pos=low.find(marker)
        if pos >= 0: return s[max(0,pos-160):pos+1000]
    return s[:700] + (' ... [middle omitted] ... ' + s[-700:] if len(s)>1500 else '')
examples['added_excerpt'] = examples.diff_html.map(lambda s: useful_excerpt(diff_lines(s,'addedline')))
examples['removed_excerpt'] = examples.diff_html.map(lambda s: useful_excerpt(diff_lines(s,'deletedline')))
examples['diff_url'] = examples.apply(lambda r: 'https://en.wikipedia.org/w/index.php?title='+quote(r.article_title.replace(' ','_'))+'&diff='+str(int(r.revision_id))+'&oldid='+str(int(r.parent_id)),axis=1)
def classify_diff(r):
    added=(r.added_excerpt or '').lower(); removed=(r.removed_excerpt or '').lower(); summary_text=(r.edit_comment or '').lower()
    markup=str(r.diff_html).lower()
    spam_markers=['fuck!!!','you are really fat','aaaaaaaaaaaaaaaa','bsadf82q','sadfsadf']
    if 'revert' in summary_text or 'undo' in summary_text or 'rollback' in summary_text:
        if any(x in removed for x in spam_markers) and not any(x in added for x in spam_markers): return 'Verified reversion removing spam-like text'
        return 'Reversion candidate; paired diff removes/changes prior content'
    if any(x in markup for x in spam_markers):
        return 'Vandalism/spam-like text removed' if any(x in removed for x in spam_markers) else 'Vandalism/spam-like text added; not historical content'
    if 'missing information' in added or '{{very long' in added: return 'Maintenance-template addition; flags coverage/length, does not add historical prose'
    if 'encyclopedia' in added or 'journal' in added or 'online]' in added: return 'Reference or bibliography addition; source quality needs review'
    if removed and not added: return 'Text removed; factual reason requires source verification'
    if added or removed: return 'Text or list changed; interpretation requires line-by-line review'
    return 'No changed line text extracted; inspect the linked diff'
examples['change_classification'] = examples.apply(classify_diff,axis=1)
examples['reviewer_notes'] = examples.change_classification
save_table(examples.drop(columns='diff_html'),'step05_revision_case_studies.csv')
display(examples[['article_title','timestamp','revision_id','parent_id','size_change_bytes','change_classification','added_excerpt','removed_excerpt','diff_url']])


,article_title,timestamp,revision_id,parent_id,size_change_bytes,change_classification,added_excerpt,removed_excerpt,diff_url
0,Russian Revolution,2005-10-25 19:23:47+00:00,26464652,26464358,-3537205.0,Vandalism/spam-like text removed,{{History_of_Russia}} | The '''Russian Revolut...,. you are really fat and like lots of men. pet...,https://en.wikipedia.org/w/index.php?title=Rus...
1,World War I,2005-12-08 13:02:16+00:00,30587516,30567121,2642330.0,Vandalism/spam-like text added; not historical...,FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK...,==Causes== | [[Image:gavrilloprincip.jpg|thumb...,https://en.wikipedia.org/w/index.php?title=Wor...
2,Assassination of John F. Kennedy,2007-02-19 21:05:08+00:00,109385280,109384443,-1716907.0,Vandalism/spam-like text removed,,bsadf82q702\tq35wbdokfosdaiyft09q8345t045t01lh...,https://en.wikipedia.org/w/index.php?title=Ass...
3,World War II,2005-12-23 03:19:58+00:00,32441095,32440959,1524224.0,Vandalism/spam-like text added; not historical...,AAAAAAAAAAAAAAAAaAAAAAAAAAAAAAAAAaAAAAAAAAAAAA...,==Participants== | [[Image:Yalta Conference.jp...,https://en.wikipedia.org/w/index.php?title=Wor...
4,September 11 attacks,2026-11-08 04:05:24+00:00,1384420834,1384180652,183.0,Text or list changed; interpretation requires ...,* [[Air India Flight 182]] – 1985 aircraft bom...,* [[Air India Flight 182]] | * [[Tokyo subway ...,https://en.wikipedia.org/w/index.php?title=Sep...
5,Vietnam War,2026-11-06 21:54:36+00:00,1384155769,1384088934,174.0,Maintenance-template addition; flags coverage/...,{{missing information|[[Võ Nguyên Giáp|Võ Nguy...,,https://en.wikipedia.org/w/index.php?title=Vie...
6,Emancipation Proclamation,2026-11-10 06:27:03+00:00,1384830413,1384718356,72.0,Reference or bibliography addition; source qua...,"* Egerton, Douglas R. {{""'}}A Measure Alike Mi...","* Egerton, Douglas R. {{""'}}A Measure Alike Mi...",https://en.wikipedia.org/w/index.php?title=Ema...
7,American Civil War,2026-11-09 17:51:56+00:00,1384701279,1384162791,-52.0,Text removed; factual reason requires source v...,,"* 50,000 free civilians died<ref name=""Nofi200...",https://en.wikipedia.org/w/index.php?title=Ame...


In [5]:
# Fetch the follow-up revisions to verify the suspected rapid World War I reverts.
sequence_ids=[30587560,30587586,30587594]
sequence_rows=df[df.revision_id.isin(sequence_ids)].copy()
sequence_rows['candidate_reason']='World War I suspected spam/reversion sequence; paired with 30587516'
candidates=pd.concat([candidates,sequence_rows]).drop_duplicates('revision_id',keep='first')
save_table(candidates[['article_title','timestamp','revision_id','parent_id','username','edit_comment','article_size_bytes','size_change_bytes','candidate_reason']], 'step05_significant_revision_candidates.csv')
extra=[]
for row in sequence_rows.itertuples():
    rec={'article_title':row.article_title,'timestamp':str(row.timestamp),'revision_id':int(row.revision_id),'parent_id':int(row.parent_id),'edit_comment':row.edit_comment,'size_change_bytes':row.size_change_bytes,'retrieval_status':'not attempted','diff_html':''}
    try:
        response=requests.get(API,params={'action':'compare','fromrev':rec['parent_id'],'torev':rec['revision_id'],'format':'json','formatversion':2},headers={'User-Agent':'WikipediaDigitalArchaeologyResearch/1.0 (educational analysis)'},timeout=5)
        response.raise_for_status(); payload=response.json(); comp=payload.get('compare',{})
        if comp.get('body') is not None: rec['diff_html']=comp['body']; rec['retrieval_status']='retrieved'
        else: rec['retrieval_status']='API returned no diff: '+str(payload.get('errors',payload))[:400]
    except Exception as exc: rec['retrieval_status']=f'retrieval failed: {type(exc).__name__}: {exc}'
    extra.append(rec); time.sleep(1.0)
diffs=pd.concat([diffs,pd.DataFrame(extra)]).drop_duplicates('revision_id',keep='last')
save_table(diffs,'step05_revision_diffs_raw.csv')
examples=diffs.copy()
examples['added_excerpt']=examples.diff_html.map(lambda s: diff_lines(s,'addedline')).str.slice(0,1400)
examples['removed_excerpt']=examples.diff_html.map(lambda s: diff_lines(s,'deletedline')).str.slice(0,1400)
examples['diff_url']=examples.apply(lambda r:'https://en.wikipedia.org/w/index.php?title='+quote(r.article_title.replace(' ','_'))+'&diff='+str(int(r.revision_id))+'&oldid='+str(int(r.parent_id)),axis=1)
examples['change_classification']=examples.apply(classify_diff,axis=1)
examples['reviewer_notes']=examples.change_classification
save_table(examples.drop(columns='diff_html'),'step05_revision_case_studies.csv')
display(examples[['article_title','timestamp','revision_id','parent_id','change_classification','added_excerpt','removed_excerpt','diff_url']])


,article_title,timestamp,revision_id,parent_id,change_classification,added_excerpt,removed_excerpt,diff_url
0,Russian Revolution,2005-10-25 19:23:47+00:00,26464652,26464358,Vandalism/spam-like text removed,{{History_of_Russia}} | The '''Russian Revolut...,. you are really fat and like lots of men. pet...,https://en.wikipedia.org/w/index.php?title=Rus...
1,World War I,2005-12-08 13:02:16+00:00,30587516,30567121,Vandalism/spam-like text added; not historical...,FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK...,==Causes== | [[Image:gavrilloprincip.jpg|thumb...,https://en.wikipedia.org/w/index.php?title=Wor...
2,Assassination of John F. Kennedy,2007-02-19 21:05:08+00:00,109385280,109384443,Vandalism/spam-like text removed,,bsadf82q702\tq35wbdokfosdaiyft09q8345t045t01lh...,https://en.wikipedia.org/w/index.php?title=Ass...
3,World War II,2005-12-23 03:19:58+00:00,32441095,32440959,Vandalism/spam-like text added; not historical...,AAAAAAAAAAAAAAAAaAAAAAAAAAAAAAAAAaAAAAAAAAAAAA...,==Participants== | [[Image:Yalta Conference.jp...,https://en.wikipedia.org/w/index.php?title=Wor...
4,September 11 attacks,2026-11-08 04:05:24+00:00,1384420834,1384180652,Text or list changed; interpretation requires ...,* [[Air India Flight 182]] – 1985 aircraft bom...,* [[Air India Flight 182]] | * [[Tokyo subway ...,https://en.wikipedia.org/w/index.php?title=Sep...
5,Vietnam War,2026-11-06 21:54:36+00:00,1384155769,1384088934,Maintenance-template addition; flags coverage/...,{{missing information|[[Võ Nguyên Giáp|Võ Nguy...,,https://en.wikipedia.org/w/index.php?title=Vie...
6,Emancipation Proclamation,2026-11-10 06:27:03+00:00,1384830413,1384718356,Reference or bibliography addition; source qua...,"* Egerton, Douglas R. {{""'}}A Measure Alike Mi...","* Egerton, Douglas R. {{""'}}A Measure Alike Mi...",https://en.wikipedia.org/w/index.php?title=Ema...
7,American Civil War,2026-11-09 17:51:56+00:00,1384701279,1384162791,Text removed; factual reason requires source v...,,"* 50,000 free civilians died<ref name=""Nofi200...",https://en.wikipedia.org/w/index.php?title=Ame...
0,World War I,2005-12-08 13:02:52+00:00,30587560,30587516,Verified reversion removing spam-like text,==Causes== | [[Image:gavrilloprincip.jpg|thumb...,FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK...,https://en.wikipedia.org/w/index.php?title=Wor...
1,World War I,2005-12-08 13:03:19+00:00,30587586,30587560,Vandalism/spam-like text added; not historical...,FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK!!!FUCK...,==Causes== | [[Image:gavrilloprincip.jpg|thumb...,https://en.wikipedia.org/w/index.php?title=Wor...
